In [ ]:
import jax
import jax.numpy as jnp
from jax import random
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import plotly.express as px

In [ ]:
#from src.jax_resnet.model import init_params, batched_forward, sample_dropout_mask, batched_forward_dropout, FiniteResNetParams, batched_forward_track, batched_forward_dropout_track
from src.jax_resnet.model import init_params, FiniteResNetParams, relu, batched_forward_track
from src.jax_resnet.training import train_scan_jit, train_dropout_scan_jit, train_ram_scan_jit, train_scan_ce_jit, train_dropout_scan_ce_jit, train_ram_scan_ce_jit

In [ ]:
from src.utils import make_dataset_mnist

In [ ]:
d_in, d_out, seed = 784, 2, 42
N=100
X_train, Y_train, X_test, Y_test = make_dataset_mnist(N=N, seed=seed, digits=[4,7])

In [ ]:
X_train.shape, Y_train.shape, X_test.shape, Y_test.shape

In [ ]:
configurations = [   
    (10, 4, 2),
    (10, 4, 4),
    (10, 4, 8),
    # (10, 4, 4),
    # (10, 40, 4),
    # (10, 40, 40),
    # (10, 400, 40),
    # (10, 400, 400),
    # (10, 2000, 15),
    # Add more as needed
]

print("Configurations to load:", configurations)


In [ ]:
def compute_parameter_distance_metrics(final_params_noisy, final_params_ref, histories_noisy, histories_ref, variant_name=None):
    """
    Compute parameter distance metrics between noisy (DO/RaM) and reference (GD) solutions.
    
    Args:
        final_params_noisy: params from dropout/ram training (dict of lists or single params)
        final_params_ref: reference params (e.g., from GD)
        histories_noisy: histories from dropout/ram training (dict of lists or single history)
        histories_ref: reference history (e.g., from GD)
        variant_name: variant identifier for printing
    
    Returns:
        Dictionary with Delta_U, Delta_V, Delta_h metrics (averaged over repetitions if applicable)
    """
    
    # Handle both single runs and multiple repetitions
    if isinstance(final_params_noisy, list):
        num_reps = len(final_params_noisy)
        is_repeated = True
    else:
        num_reps = 1
        final_params_noisy = [final_params_noisy]
        histories_noisy = [histories_noisy]
        is_repeated = False
    
    all_delta_u = []
    all_delta_v = []
    all_delta_h = []
    
    for rep_idx in range(num_reps):
        params_noisy = final_params_noisy[rep_idx]
        params_ref = final_params_ref[rep_idx]
        history_noisy = histories_noisy[rep_idx]
        history_ref = histories_ref[rep_idx]
        
        U_noisy = np.asarray(params_noisy.U)  # (L, M, D)
        V_noisy = np.asarray(params_noisy.V)  # (L, M, D)
        
        U_ref = np.asarray(params_ref.U)      # (L, M, D)
        V_ref = np.asarray(params_ref.V)      # (L, M, D)
        
        # Compute Delta_U: max_l sqrt(1/M * sum_j ||U_noisy[l,j,:] - U_ref[l,j,:]||_2^2)
        delta_u_per_layer = []
        for l in range(U_noisy.shape[0]):
            diff = U_noisy[l, :, :] - U_ref[l, :, :]  # (M, D)
            rms = np.sqrt(np.mean(np.mean(diff**2, axis=1)))  # RMS over particles
            delta_u_per_layer.append(rms)
        delta_u = np.mean(delta_u_per_layer)
        
        # Compute Delta_V: max_l sqrt(1/M * sum_j ||V_noisy[l,j,:] - V_ref[l,j,:]||_2^2)
        delta_v_per_layer = []
        for l in range(V_noisy.shape[0]):
            diff = V_noisy[l, :, :] - V_ref[l, :, :]  # (M, D)
            rms = np.sqrt(np.mean(np.mean(diff**2, axis=1)))
            delta_v_per_layer.append(rms)
        delta_v = np.mean(delta_v_per_layer)
        
        # Compute Delta_h: max_{i,l} ||h_noisy[l,i,:] - h_ref[l,i,:]||_2
        h_noisy = np.asarray(history_noisy['test_h'])  # (n, L, D)
        h_ref = np.asarray(history_ref['test_h'])      # (n, L, D)
        
        diff_h = h_noisy - h_ref  # (n, L, D)
        l2_norms_h = np.linalg.norm(diff_h, axis=2)  # (n, L)
        delta_h = np.mean(l2_norms_h)
        
        all_delta_u.append(delta_u)
        all_delta_v.append(delta_v)
        all_delta_h.append(delta_h)
    
    # Average over repetitions if applicable
    result = {
        'Delta_U': np.mean(all_delta_u),
        'Delta_V': np.mean(all_delta_v),
        'Delta_h': np.mean(all_delta_h),
    }
    
    if is_repeated:
        result['Delta_U_std'] = np.std(all_delta_u)
        result['Delta_V_std'] = np.std(all_delta_v)
        result['Delta_h_std'] = np.std(all_delta_h)
    
    return result

In [ ]:
import pickle
# Load checkpoints and compute metrics for all configurations
tau_multi = 0.4
q_multi = 0.5
n_steps_multi = 200
lr_in_multi, lr_out_multi = 0.0, 0.0
batch_size_multi = 64
last_particle_single_source_multi = True
num_repetitions_multi = 5
LOOP_SEED_multi = 48
d_in_multi, d_out_multi, seed_multi = 784, 2, 42
N_multi = 100

results_multi = []

for D_config, L_config, M_config in configurations:
    setting_str = f'L{L_config}_M{M_config}_D{D_config}_tau{tau_multi}_q{q_multi}_nsteps{n_steps_multi}_din{d_in_multi}_dout{d_out_multi}_seed{seed_multi}_N{N_multi}_numrepetitions{num_repetitions_multi}_loopseed{LOOP_SEED_multi}_batchsize{batch_size_multi}_lpss{last_particle_single_source_multi}'
    
    print(f"\nLoading configuration: D={D_config}, L={L_config}, M={M_config}")
    try:
        with open(f'data/mnist/final_params_do_{setting_str}.pkl', 'rb') as f:
            final_params_do_config = pickle.load(f)
        with open(f'data/mnist/histories_do_{setting_str}.pkl', 'rb') as f:
            histories_do_config = pickle.load(f)
        with open(f'data/mnist/final_params_ram_{setting_str}.pkl', 'rb') as f:
            final_params_ram_config = pickle.load(f)
        with open(f'data/mnist/histories_ram_{setting_str}.pkl', 'rb') as f:
            histories_ram_config = pickle.load(f)
        with open(f'data/mnist/final_params_gd_{setting_str}.pkl', 'rb') as f:
            final_params_gd_config = pickle.load(f)
        with open(f'data/mnist/histories_gd_{setting_str}.pkl', 'rb') as f:
            histories_gd_config = pickle.load(f)
        
        # Compute metrics for each variant
        for variant in ['full_unit_dropout', 'stochastic_depth', 'single_source_M']:
            metrics_do_vs_ram = compute_parameter_distance_metrics(
                final_params_do_config[variant],
                final_params_ram_config[variant],
                histories_do_config[variant],
                histories_ram_config[variant],
                variant_name=f"dropout:{variant}"
            )
            
            results_multi.append({
                'D': D_config,
                'L': L_config,
                'M': M_config,
                'M*L': L_config * M_config,
                'variant': variant,
                'method': 'dropout',
                'Delta_U': metrics_do_vs_ram['Delta_U'],
                'Delta_U_std': metrics_do_vs_ram.get('Delta_U_std', 0),
                'Delta_V': metrics_do_vs_ram['Delta_V'],
                'Delta_V_std': metrics_do_vs_ram.get('Delta_V_std', 0),
                'Delta_h': metrics_do_vs_ram['Delta_h'],
                'Delta_h_std': metrics_do_vs_ram.get('Delta_h_std', 0),
            })
            
            print(f"  {variant}:")
            print(f"    DO: ΔU = {metrics_do_vs_ram['Delta_U']:.4e}, ΔV = {metrics_do_vs_ram['Delta_V']:.4e}")

    except FileNotFoundError as e:
        print(f"  WARNING: Could not load {setting_str}: {e}")

# Convert to DataFrame for easier plotting
import pandas as pd
df_results = pd.DataFrame(results_multi)
print("\nResults summary:")
print(df_results)


In [ ]:
df_results

In [ ]:
def plot_metric_vs_ML(
    df,
    metric='Delta_U',
    variants=None,
    methods=None,
    log_x=False,
    log_y=False,
    fit_ml_inv_sqrt=False,
    fit_ml_bounds=None,
    fit_color='0.35',
    fit_linestyle='--',
    fit_alpha=0.9,
    figsize=(7, 5),
    show=True,
):
    """Plot `metric` vs M*L on a single axis for all selected variants and methods.

    Args:
        df: pandas DataFrame containing columns ['D','L','M','M*L','variant','method',metric]
        metric: string key to plot; must be one of 'Delta_U','Delta_V','Delta_h'
        variants: list of variant names to include (defaults to all in df)
        methods: list of methods to include (defaults to all in df)
        log_x, log_y: whether to use log scale for axes
        fit_ml_inv_sqrt: if True/False, apply to all variants. Can also be:
                         - list of bools (one per variant, in same order as variants)
                         - dict mapping variant names to bools
        fit_ml_bounds: (lower, upper) bounds for M*L values to include in fit. Can be:
                       - None (use all data)
                       - tuple (lower, upper) to apply to all variants
                       - dict mapping variant names to (lower, upper) tuples
                       - list of (lower, upper) tuples, one per variant
        fit_color: color used for the fit line when the series color is unavailable
        fit_linestyle: linestyle used for the fit curve
        fit_alpha: alpha used for the fit curve
        figsize: tuple width,height for the figure
        show: whether to plt.show() the figure

    Returns:
        matplotlib Figure
    """
    if metric not in ('Delta_U', 'Delta_V', 'Delta_h'):
        raise ValueError("metric must be one of 'Delta_U','Delta_V','Delta_h'")

    if variants is None:
        variants = list(df['variant'].unique())
    if methods is None:
        methods = list(df['method'].unique())

    # Normalize fit_ml_inv_sqrt to a dict mapping variant -> bool
    if isinstance(fit_ml_inv_sqrt, bool):
        fit_per_variant = {v: fit_ml_inv_sqrt for v in variants}
    elif isinstance(fit_ml_inv_sqrt, dict):
        fit_per_variant = fit_ml_inv_sqrt
    elif isinstance(fit_ml_inv_sqrt, (list, tuple)):
        if len(fit_ml_inv_sqrt) != len(variants):
            raise ValueError(f"fit_ml_inv_sqrt list length ({len(fit_ml_inv_sqrt)}) must match variants length ({len(variants)})")
        fit_per_variant = {v: f for v, f in zip(variants, fit_ml_inv_sqrt)}
    else:
        raise TypeError("fit_ml_inv_sqrt must be bool, list, tuple, or dict")

    # Normalize fit_ml_bounds to a dict mapping variant -> (lower, upper)
    if fit_ml_bounds is None:
        bounds_per_variant = {v: (None, None) for v in variants}
    elif isinstance(fit_ml_bounds, tuple) and len(fit_ml_bounds) == 2:
        bounds_per_variant = {v: fit_ml_bounds for v in variants}
    elif isinstance(fit_ml_bounds, dict):
        bounds_per_variant = {v: fit_ml_bounds.get(v, (None, None)) for v in variants}
    elif isinstance(fit_ml_bounds, (list, tuple)):
        if len(fit_ml_bounds) != len(variants):
            raise ValueError(f"fit_ml_bounds list length ({len(fit_ml_bounds)}) must match variants length ({len(variants)})")
        bounds_per_variant = {v: b for v, b in zip(variants, fit_ml_bounds)}
    else:
        raise TypeError("fit_ml_bounds must be None, tuple, dict, or list")

    fig, ax = plt.subplots(1, 1, figsize=figsize)

    metric_label_map = {
        'Delta_U': r'$\Delta U$',
        'Delta_V': r'$\Delta V$',
        'Delta_h': r'$\Delta h$',
    }

    variant_display_map = {
        'full_unit_dropout': 'full-unit',
        'stochastic_depth': 'stochastic-depth',
        'single_source_M': 'depth_shared',
    }
    variant_color_map = {
        'full_unit_dropout': 'tab:blue',
        'stochastic_depth': 'tab:green',
        'single_source_M': 'tab:purple',
    }

    markers = ['o', 's', '^', 'd', 'x', 'P', 'v']
    has_any = False
    for v_idx, variant in enumerate(variants):
        variant_color = variant_color_map.get(variant, fit_color)
        variant_label = variant_display_map.get(variant, variant)
        should_fit = fit_per_variant.get(variant, False)
        ml_lower, ml_upper = bounds_per_variant.get(variant, (None, None))
        has_any = False
        for m_idx, method in enumerate(methods):
            subset = df[(df['variant'] == variant) & (df['method'] == method)].copy()
            if subset.empty:
                continue
            has_any = True
            subset = subset.sort_values('M*L')
            x = np.asarray(subset['M*L'].values)
            y = np.asarray(subset[metric].values)
            yerr = None
            std_col = metric + '_std'
            if std_col in subset.columns:
                yerr = subset[std_col].values

            marker = markers[(v_idx + m_idx) % len(markers)]
            label = f'{variant_label} / {method}'

            ax.errorbar(
                x,
                y,
                yerr=yerr,
                marker=marker,
                color=variant_color,
                linestyle='-',
                label=label,
                capsize=4,
                markersize=6,
            )

            if should_fit:
                fit_mask = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
                if ml_lower is not None:
                    fit_mask = fit_mask & (x >= ml_lower)
                if ml_upper is not None:
                    fit_mask = fit_mask & (x <= ml_upper)
                if np.count_nonzero(fit_mask) >= 2:
                    x_fit = x[fit_mask]
                    y_fit = y[fit_mask]
                    log_c = np.mean(np.log(y_fit) + 0.5 * np.log(x_fit))
                    c = float(np.exp(log_c))
                    x_line = np.geomspace(x_fit.min(), x_fit.max(), 200)
                    y_line = c * x_line ** (-0.5)
                    ax.plot(
                        x_line,
                        y_line,
                        color=variant_color,
                        linestyle=fit_linestyle,
                        alpha=fit_alpha,
                        linewidth=1.8,
                        label=f'{variant_label} fit $(M L)^{{-1/2}}$',
                    )

    if not has_any:
        ax.text(0.5, 0.5, 'no data', ha='center', va='center', transform=ax.transAxes)

    ax.set_xlabel('M × L')
    ax.set_ylabel(metric_label_map[metric])
    ax.set_title(f'{metric_label_map[metric]} vs. effective model width (M x L)')
    ax.grid(alpha=0.25, which='both')
    ax.legend()

    if log_x:
        ax.set_xscale('log')
    if log_y:
        ax.set_yscale('log')

    fig.tight_layout()
    if show:
        plt.show()
    return fig


In [ ]:
figU = plot_metric_vs_ML(df_results, metric='Delta_U', log_x=True, log_y=True, fit_ml_inv_sqrt=False)

In [ ]:
#figU.savefig("plots/Delta_U_vs_ML.pdf")

In [ ]:
figV = plot_metric_vs_ML(df_results, metric='Delta_V', log_x=True, log_y=True, fit_ml_inv_sqrt=False)

In [ ]:
#figV.savefig("plots/Delta_V_vs_ML.pdf")

In [ ]:
figh = plot_metric_vs_ML(df_results, metric='Delta_h', log_x=True, log_y=True, fit_ml_inv_sqrt=[True, False, False], fit_ml_bounds=(10, 100))

In [ ]:
#figh.savefig("plots/Delta_h_vs_ML.pdf")